In [1]:
from __future__ import annotations
import os
import json
import importlib
import argparse
from functools import partial
import pprint

import random
import numpy as np

import torch

from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL)

# from core import mcts_search_extra_v21
from core import mcts_search_test_v31
# from core import mcts_search_extra_v61
# from core import mcts_search_extra_v71

from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k

from sal.utils.score import aggregate_scores

# from core.llm_engine import rm_engine
# from core.llms import rm_generate
# import logging
# logging.basicConfig(format='%(message)s', level=logging.ERROR)

INFO 10-31 06:36:24 [__init__.py:244] Automatically detected platform cuda.


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
llm_total_gpu = 0.4
llm_gpu_memory_utilization = 0.2
# llm_vllm = LLM(
#     model = llm_dir,
#     tensor_parallel_size=1,
#     gpu_memory_utilization = 0.7,  # Utilize 50% of GPU memory
#     # enable_prefix_caching=True,  # V100 doesn't support enable_prefix_caching 
#     # enable_chunked_prefill=False, # and enable_chunked_prefill
#     max_model_len = 5000,
#     dtype = "float16",
#     seed = config.seed)

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)

print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))



INFO 10-31 06:36:47 [config.py:823] This model supports multiple tasks: {'embed', 'classify', 'generate', 'score', 'reward'}. Defaulting to 'generate'.
WARNING 10-31 06:36:47 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 10-31 06:36:47 [arg_utils.py:1642] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 10-31 06:36:47 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 10-31 06:36:47 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_par

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 10-31 06:36:56 [default_loader.py:272] Loading weights took 6.54 seconds
INFO 10-31 06:36:57 [model_runner.py:1203] Model loading took 2.3185 GiB and 6.842521 seconds
INFO 10-31 06:36:58 [worker.py:294] Memory profiling takes 0.72 seconds
INFO 10-31 06:36:58 [worker.py:294] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.20) = 6.35GiB
INFO 10-31 06:36:58 [worker.py:294] model weights take 2.32GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 2.75GiB.
INFO 10-31 06:36:58 [executor_base.py:113] # cuda blocks: 5631, # CPU blocks: 32768
INFO 10-31 06:36:58 [executor_base.py:118] Maximum concurrency for 5000 tokens per request: 18.02x
INFO 10-31 06:37:05 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 7.92 seconds
#--- memory: 5.07647705078125


In [5]:
llm_vllm_embeds = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    task="embed",
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

WARNING 10-31 06:37:05 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 10-31 06:37:05 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 10-31 06:37:05 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_pr

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 10-31 06:37:08 [default_loader.py:272] Loading weights took 1.29 seconds
INFO 10-31 06:37:08 [model_runner.py:1203] Model loading took 2.3029 GiB and 1.335798 seconds
#--- memory: 7.379390716552734


In [6]:
prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

#--- memory: 22.336918354034424


In [12]:
from collections import deque

def level_order(root, level_thres):
    res = []
    queue = deque([root]) # start with root

    # level 0: root 
    res.append([root.visit_count()])

    while queue:
        level_size = len(queue)
        level_groups = []

        for _ in range(level_size):
            node = queue.popleft()
            children = node.children
            if children:
                level_groups.append([child.visit_count() for child in children])

                for child in children:
                    queue.append(child)

        if level_groups:
            res.append(level_groups)

    return res
    


In [13]:
def tree_to_dict(node):
    return {
        "tag": node.tag,
        "depth": node.depth,
        "q_value": f"{node.q_value():0.5f}",
        "puct_value": f"{node.puct(cpuct=2):0.5f}",
        "nvisits": f"{node.visit_count()} ({node.visit_percent():0.2f})",
        "is_terminal": node.is_terminal,
        "text": node.state["step"],
        "children": [tree_to_dict(child) for child in node.children]
        
    }

In [9]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [10]:
stop

NameError: name 'stop' is not defined

In [14]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 40            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0
config.date_string = "Aug 1 2025"

config.num_batches = 2
config.step_budget = config.num_batches*config.max_depths 
config.num_phases = 100

config.lam = 0.01
config.normalize_embeds = True

config.cpuct = 2
config.ds_beta = 1.0
config.ds_alpha = 100.0
config.use_ppl = True
config.negative_reward = 0

# config.version = "v51"

In [15]:
level = 4                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
# num_questions = 1
trial_idx = 2
print(f"num_questions = {num_questions}")
print(f"trial = {trial_idx}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 79
batch_of_questions = [data_by_levels[level][q_idx]['problem']]
print(batch_of_questions)

num_questions = 128
trial = 2
['Let $\\mathbf{a},$ $\\mathbf{b},$ $\\mathbf{c}$ be three vectors such that\n\\[\\mathbf{a} \\times \\mathbf{b} = \\begin{pmatrix} 6 \\\\ -7 \\\\ 3 \\end{pmatrix}, \\quad \\mathbf{a} \\times \\mathbf{c} = \\begin{pmatrix} 4 \\\\ 7 \\\\ 2 \\end{pmatrix}, \\quad \\mathbf{b} \\times \\mathbf{c} = \\begin{pmatrix} 1 \\\\ -7 \\\\ 18 \\end{pmatrix}.\\]Compute $(2 \\mathbf{b} - \\mathbf{a}) \\times (3 \\mathbf{c} + \\mathbf{a}).$']


In [16]:
importlib.reload(mcts_search_test_v31)

config.max_depths = 40            # max depths, after reaching max_depth then terminate search 
config.num_batches = 2
config.step_budget = config.num_batches*config.max_depths 

config.num_phases = 50

logging.fatal(f"trial = {trial_idx}")

np.random.seed(100000+trial_idx)
random.seed(100000+trial_idx)
torch.manual_seed(100000+trial_idx)
torch.cuda.manual_seed(100000+trial_idx)

for _, question in enumerate(batch_of_questions):
    agent = mcts_search_test_v31.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, step_cnt, p, ndepths_arr, cnt_node_max_depth = \
        mcts_search_test_v31.mcts_search(question, agent, config, llm_vllm, prm)

trial = 2

-> p = 0

-> d = 0
0.1
   q-value = 0.1824
   u-value = 0.0000
   puct = 0.1824
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2
   q-value = 0.0341
   u-value = 0.0000
   puct = 0.0341
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.3
   q-value = 0.1896
   u-value = 0.0000
   puct = 0.1896
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.4
   q-value = 0.0252
   u-value = 0.0000
   puct = 0.0252
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
selected_child = 0.3
self.cnt_sel_div = 0/1 (0.00)
self.cnt_sel_expl = 0/1 (0.00)
step_cnt = 1

-> d = 1
0.3.1
   q-value = 0.1481
   u-value = 0.0000
   puct = 0.1481
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.3.2
   q-value = 0.1624
   u-value = 0.0000
   puct = 0.1624
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.3.3
   q-value = 0.2310
   u-value = 0.0000
   puct = 0.2310
   nvisit = 1.00
   parent.nvisit = 1.00
   is_t

In [17]:
version = "v31"
tree_dict = tree_to_dict(agent.root)
# pprint.pprint(tree_dict, indent=2)
with open(f"zlogs/tree_level-{level}--q-{q_idx}--{version}--bs-{config.n}--d-{config.max_depths}--b-{config.step_budget}--trial-{trial_idx}.json", 'w', encoding = 'utf-8') as fout:
    json.dump(tree_dict, fout)
    fout.write('\n') 

In [18]:
print(f"step_cnt = {step_cnt}")
print(f"total_nphases = {p}")
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
# print(f"nvisits = {c_nvisits}")
print(f"nvisits = ")
pprint.pprint(c_nvisits, compact=True)
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")
print(f"cnt_node_max_depth = {agent.cnt_node_max_depth}")

print(f"cnt_sel_div = {agent.cnt_sel_div}/{agent.cnt_sel_all} ({agent.cnt_sel_div/agent.cnt_sel_all:0.2f})")
print(f"cnt_sel_expl = {agent.cnt_sel_expl}/{agent.cnt_sel_all} ({agent.cnt_sel_expl/agent.cnt_sel_all:0.2f})")
print(f"selection_gap_diverse = {np.array(agent.gap_div)}")
print(f"selection_gap_all = {np.array(agent.gap_all)}")


step_cnt = 80
total_nphases = 7
ncomps = 9
nvisits = 
[[8], [[3, 3, 3, 2]], [[2, 1, 2, 1], [1, 2, 2, 1], [1, 1, 2, 2], [2]],
 [[1, 1, 1, 2], [2, 1, 1, 1], [2, 1], [2, 1, 1, 1], [1, 1, 1, 1], [1, 1, 2, 1],
  [1, 2, 1, 1]],
 [[2, 1, 1, 1], [1, 1, 1, 2], [1, 2], [1, 2, 1, 1], [1, 1, 1, 1], [1, 1, 2, 1],
  [2, 1, 1, 1]],
 [[2, 1, 1, 1], [1, 1, 1, 2], [1, 2, 1, 1], [2, 1, 1, 1], [1, 1, 1, 1],
  [1, 1, 1, 2], [1, 2, 1, 1]],
 [[2], [2, 1, 1, 1], [1, 1, 1, 2], [1, 1, 1, 1], [1, 1, 1, 2], [2, 1, 1, 1]],
 [[1, 2, 1, 1], [1, 2, 1, 1], [1, 1, 1, 1], [1, 1, 1, 2], [1, 1, 2, 1]],
 [[1, 1, 2, 1], [1, 2, 1, 1], [1, 1, 1, 1], [1, 1, 1, 2], [2, 1, 1, 1]],
 [[2, 1, 1, 1], [2, 1], [1, 1, 1, 1], [2, 1, 1, 1], [1, 1, 2, 1]],
 [[1, 1, 2, 1], [1, 1, 1, 1], [1, 2, 1, 1], [2, 1, 1, 1]],
 [[1, 2], [1, 1, 1, 1], [2], [2, 1, 1, 1]],
 [[1, 1, 1, 2], [1, 1, 1], [1, 1, 1, 2]],
 [[1, 1, 2, 1], [1, 1, 1, 1], [1, 1, 1, 2]],
 [[1, 1, 2, 1], [1, 1, 1, 1], [1, 1, 2, 1]], [[1, 2, 1, 1], [1, 1, 1, 1], [2]],
 [[2, 1], [1, 1, 

In [ ]:
for com_idx, comp in enumerate(agent_completions):
    print(f"\n-> com_idx = {com_idx}")
    print(comp)
